# 第 14 章：RAG Baseline

对应新版 `roadmap.md` 主线入口。

**核心问题**：模型参数不是数据库，怎么让模型回答前先查资料？

**本章关注**：chunking, embedding search, top-k, citation span, 无依据拒答, prompt injection 防御。

**轻量实验**：用关键词重叠模拟检索，找不到依据时输出 unknown。

> 说明：本 notebook 只做最小可运行观察，不做大型训练；如果后续要接入仓库内 API，请先确认 API 已存在。


In [ ]:
chunks = [
    {"source_id": "law-001", "span_id": "s1", "text": "合同违约金应结合实际损失判断。"},
    {"source_id": "med-001", "span_id": "s1", "text": "胸痛伴呼吸困难属于需要及时就医的危险信号。"},
]
query = "合同 违约金 风险"
query_terms = query.split()

scored = []
for chunk in chunks:
    score = sum(term in chunk["text"] for term in query_terms)
    scored.append((score, chunk))
scored.sort(reverse=True, key=lambda x: x[0])

best_score, best_chunk = scored[0]
answer = {
    "answer": "unknown" if best_score == 0 else "请基于引用片段进一步人工判断。",
    "citations": [] if best_score == 0 else [
        {
            "source_id": best_chunk["source_id"],
            "span_id": best_chunk["span_id"],
            "support_level": "partial",
        }
    ],
    "needs_human_review": True,
}
print(answer)


## 学习观察

运行上面的最小实验后，建议记录三点：

1. 哪个输入或配置最影响输出？
2. 这个 toy 实验和本章核心问题之间的对应关系是什么？
3. 如果要进入 `src/` 或真实模型实现，还缺哪些已确认的 API、测试或数据？

本章验收时优先看能否解释：模型参数不是数据库，怎么让模型回答前先查资料？
